In [ ]:
"""
U-Net for SEM nanoparticle segmentation.
Expects preprocessed data from the preprocessing script:
    data512/images/*.png   (grayscale, 512x512)
    data512/masks/*.png    (0/255, 512x512)
"""
import os, json, random
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

IMG_DIR, MASK_DIR = "data512/images", "data512/masks"
CROP, BATCH, EPOCHS, LR = 256, 8, 50, 1e-3
SAMPLES_PER_IMAGE = 8           # random crops drawn per image per epoch
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


# ---------------------------------------------------------------- split first
names = sorted(os.listdir(IMG_DIR))
random.Random(SEED).shuffle(names)
n = len(names)
train_names = names[: int(0.70 * n)]
val_names = names[int(0.70 * n): int(0.85 * n)]
test_names = names[int(0.85 * n):]
with open("split.json", "w") as f:   # reuse the same split for the watershed baseline
    json.dump({"train": train_names, "val": val_names, "test": test_names}, f)


# ------------------------------------------------------------------- dataset
class SEMDataset(Dataset):
    def __init__(self, names, train):
        self.names, self.train = names, train
        self.reps = SAMPLES_PER_IMAGE if train else 1

    def __len__(self):
        return len(self.names) * self.reps

    def _read(self, name):
        img = cv2.imread(os.path.join(IMG_DIR, name), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        msk = (cv2.imread(os.path.join(MASK_DIR, name), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        return img, msk

    def _random_crop(self, img, msk):
        h, w = img.shape
        for _ in range(5):                       # resample mostly-empty crops
            y, x = random.randint(0, h - CROP), random.randint(0, w - CROP)
            m = msk[y:y + CROP, x:x + CROP]
            if m.any() or random.random() < 0.1:  # keep ~10% pure-background crops
                break
        return img[y:y + CROP, x:x + CROP], m

    def __getitem__(self, i):
        img, msk = self._read(self.names[i % len(self.names)])
        if self.train:
            img, msk = self._random_crop(img, msk)
            if random.random() < 0.5: img, msk = img[:, ::-1], msk[:, ::-1]
            if random.random() < 0.5: img, msk = img[::-1], msk[::-1]
            k = random.randint(0, 3)
            img, msk = np.rot90(img, k), np.rot90(msk, k)
        return (torch.from_numpy(np.ascontiguousarray(img))[None],
                torch.from_numpy(np.ascontiguousarray(msk))[None])


# --------------------------------------------------------------------- model
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]   # 32, 64, 128, 256, 512
        self.enc1 = DoubleConv(in_ch, c[0])
        self.enc2 = DoubleConv(c[0], c[1])
        self.enc3 = DoubleConv(c[1], c[2])
        self.enc4 = DoubleConv(c[2], c[3])
        self.bottleneck = DoubleConv(c[3], c[4])
        self.pool = nn.MaxPool2d(2)

        self.up4 = nn.ConvTranspose2d(c[4], c[3], 2, stride=2)
        self.dec4 = DoubleConv(c[3] * 2, c[3])
        self.up3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2)
        self.dec3 = DoubleConv(c[2] * 2, c[2])
        self.up2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2)
        self.dec2 = DoubleConv(c[1] * 2, c[1])
        self.up1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2)
        self.dec1 = DoubleConv(c[0] * 2, c[0])
        self.head = nn.Conv2d(c[0], out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)          # raw logits; sigmoid is applied in loss / eval


# ---------------------------------------------------------------------- loss
def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()


def loss_fn(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)


# ------------------------------------------------------------------- metrics
@torch.no_grad()
def evaluate(model, loader):
    """Returns (mean IoU, mean Dice, mean loss) over full images."""
    model.eval()
    ious, dices, losses = [], [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        losses.append(loss_fn(logits, y).item())
        pred = (torch.sigmoid(logits) > 0.5).float()
        inter = (pred * y).sum().item()
        union = pred.sum().item() + y.sum().item() - inter
        if union == 0:               # empty prediction and empty mask: skip
            continue
        ious.append(inter / union)
        dices.append(2 * inter / (pred.sum().item() + y.sum().item()))
    return float(np.mean(ious)), float(np.mean(dices)), float(np.mean(losses))


# ------------------------------------------------------------- visualisation
@torch.no_grad()
def save_predictions(model, names, out_dir, max_images=8):
    """Saves image | ground truth | prediction panels so you can inspect failures."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    model.eval()
    ds = SEMDataset(names, train=False)
    for i in range(min(max_images, len(ds))):
        x, y = ds[i]
        pred = (torch.sigmoid(model(x[None].to(DEVICE)))[0, 0] > 0.5).cpu().numpy()
        fig, ax = plt.subplots(1, 3, figsize=(12, 4))
        ax[0].imshow(x[0], cmap="gray"); ax[0].set_title(names[i])
        ax[1].imshow(y[0], cmap="gray"); ax[1].set_title("ground truth")
        ax[2].imshow(pred, cmap="gray"); ax[2].set_title("prediction")
        for a in ax: a.axis("off")
        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{os.path.splitext(names[i])[0]}.png"), dpi=100)
        plt.close(fig)


def save_curves(history, path="curves.png"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
    ax[1].plot(history["val_iou"], label="val IoU"); ax[1].plot(history["val_dice"], label="val Dice")
    ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); fig.savefig(path, dpi=100); plt.close(fig)


# ------------------------------------------------------------------ training
def main():
    train_loader = DataLoader(SEMDataset(train_names, True), batch_size=BATCH, shuffle=True, num_workers=2)
    val_loader = DataLoader(SEMDataset(val_names, False), batch_size=1)
    test_loader = DataLoader(SEMDataset(test_names, False), batch_size=1)

    model = UNet().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5)

    history = {"train_loss": [], "val_loss": [], "val_iou": [], "val_dice": []}
    best = 0.0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            total += loss.item() * x.size(0)
        train_loss = total / len(train_loader.dataset)
        val_iou, val_dice, val_loss = evaluate(model, val_loader)
        sched.step(val_iou)

        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["val_iou"].append(val_iou); history["val_dice"].append(val_dice)
        print(f"epoch {epoch:3d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} "
              f"| val IoU {val_iou:.4f} | val Dice {val_dice:.4f}")

        if val_iou > best:
            best = val_iou
            torch.save(model.state_dict(), "unet_best.pt")

    save_curves(history)
    json.dump(history, open("history.json", "w"))

    # final report: use the checkpoint chosen on validation, touch test once
    model.load_state_dict(torch.load("unet_best.pt", map_location=DEVICE))
    save_predictions(model, val_names, "preds_val")
    test_iou, test_dice, _ = evaluate(model, test_loader)
    print(f"TEST | IoU {test_iou:.4f} | Dice {test_dice:.4f}")
    save_predictions(model, test_names, "preds_test")


if __name__ == "__main__":
    main()
